# The junction-tree algorithm — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Cluster variables into a tree so exact inference becomes message passing again
Junction trees repair cycles by clustering variables. These examples build clique potentials on AB and BC, pass a separator message, calibrate beliefs, verify agreement with brute force, and show why cluster size matters.

In [ ]:
# 1) two cliques AB and BC joined by separator B
phiAB=np.array([[0.3,0.7],[0.8,0.2]]); phiBC=np.array([[0.9,0.1],[0.4,0.6]])
msgB=phiAB.sum(axis=0); belBC=phiBC*msgB[:,None]; belBC=belBC/belBC.sum()
plt.figure(figsize=(6,3)); plt.imshow(belBC); plt.colorbar(); plt.title('message from AB calibrates BC')
assert abs(msgB[0]-1.1)<1e-12 and abs(belBC.sum()-1)<1e-12

In [ ]:
# 2) calibrated separator beliefs agree from both sides
sep_from_BC=belBC.sum(axis=1); sep_from_AB=normalize(msgB)
plt.figure(figsize=(6,3)); plt.plot(sep_from_AB,'o-',label='AB side'); plt.plot(sep_from_BC,'x--',label='BC side'); plt.legend(); plt.title('separator B agrees after calibration')
assert np.allclose(sep_from_AB,sep_from_BC)

In [ ]:
# 3) marginal for C after collecting evidence from AB
belC=belBC.sum(axis=0); plt.figure(figsize=(6,3)); plt.bar(['C=0','C=1'],belC); plt.title(f'P(C=1)={belC[1]:.3f}')
assert abs(belC[1]-0.325)<1e-12

In [ ]:
# 4) brute-force over A,B,C matches the junction tree marginal
joint=np.zeros((2,2,2))
for a,b,c in itertools.product([0,1],repeat=3): joint[a,b,c]=phiAB[a,b]*phiBC[b,c]
br=joint.sum(axis=(0,1))/joint.sum()
plt.figure(figsize=(6,3)); plt.plot(br,'o-',label='brute C'); plt.plot(belC,'x--',label='JT C'); plt.legend(); plt.title('exact inference by cluster messages')
assert np.allclose(br,belC)

In [ ]:
# 5) treewidth cost: binary clique size grows exponentially
sizes=np.arange(1,8); entries=2**sizes
plt.figure(figsize=(6,3)); plt.semilogy(sizes,entries,'-o'); plt.xlabel('clique variables'); plt.ylabel('table entries'); plt.title('exactness costs exponential clique tables')
assert entries[-1]==128